In [15]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (20, 12),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import time
import datetime
import pytz

NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from MDP.USTFutures.USTFuturesMDP import USTFuturesMDP
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP

In [17]:
# ts =  CHI_tz.localize(datetime.datetime(2026, 3, 24, 14, 0))
ts = "live" 
symbol = "USM26"  

mdp = USTFuturesMDP(source="BARCHART_USTF-RL")

pricer = mdp.get_pricer(
    {
        "symbols": [symbol],
        "timestamp": ts,
        "include_basket": True,
    }
)[symbol]

fut = pricer.build_pricable()
fut_price = pricer.price(fut)
fut_ytm = pricer.yield_to_maturity(fut)

ctd = pricer.ctd()
ctd_meta = (ctd.meta() or {}) if ctd else {}
ctd_cusip = ctd_meta.get("cusip")

In [22]:
basket = mdp.get_delivery_basket(
    as_of=datetime.date.today() if ts == "live" else ts,
    symbol=symbol,
    source="RL_CME_TCF",
)

gross_basis_vec = pricer.gross_basis()
bnoc_vec = pricer.bnoc()

rows = []
for bond_pricer, cf, gross_basis, bnoc in zip(
    basket["basket_pricers"],
    basket["conversion_factors"],
    gross_basis_vec,
    bnoc_vec,
):
    meta = bond_pricer.meta() or {}
    clean_price = float(bond_pricer.clean_price())
    ytm = float(bond_pricer.ytm())

    rows.append(
        {
            "cusip": meta.get("cusip"),
            "label": meta.get("label"),
            "clean_price": clean_price,
            "ytm": ytm,
            "invoice_cf": float(cf),
            "gross_basis": float(gross_basis),
            "bnoc": float(bnoc),
            "is_ctd": meta.get("cusip") == ctd_cusip,
        }
    )


basis_df = pd.DataFrame(rows).sort_values(["is_ctd", "gross_basis"], ascending=[False, True]).reset_index(drop=True)

print(f"{symbol} futures price: {fut_price:.4f}")
print(f"{symbol} futures YTM:   {fut_ytm:.4f}")
print(f"CTD: {ctd_meta.get('label')} ({ctd_meta.get('cusip')})")

USM26 futures price: 112.2188
USM26 futures YTM:   4.9912
CTD: T 4 3/8 Aug 43 (912810TU2)


In [23]:
basis_df

,cusip,label,clean_price,ytm,invoice_cf,gross_basis,bnoc,is_ctd
0,912810TU2,T 4 3/8 Aug 43,93.414422,4.943,0.8283,0.408230,0.174633,True
1,912810RD2,T 3 3/4 Nov 43,85.725407,4.974,0.7602,0.363888,0.218114,False
2,912810RE0,T 3 5/8 Feb 44,84.013491,4.985,0.7448,0.380392,0.241877,False
3,912810RC4,T 3 5/8 Aug 43,84.523544,4.963,0.7491,0.408516,0.274255,False
4,912810RB6,T 2 7/8 May 43,75.941965,4.977,0.6726,0.415391,0.388701,False
5,912810TS7,T 3 7/8 May 43,87.697222,4.947,0.7773,0.427388,0.269478,False
6,912810TM0,T 4 Nov 42,89.587151,4.923,0.7941,0.432193,0.261379,False
7,912810TQ1,T 3 7/8 Feb 43,87.938204,4.935,0.7794,0.432983,0.269461,False
8,912810RH3,T 3 1/8 Aug 44,77.489086,5.012,0.6862,0.433824,0.356554,False
9,912810QZ4,T 3 1/8 Feb 43,79.214103,4.955,0.7015,0.443540,0.380655,False


In [24]:
cash_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-RL")
pricers = cash_mdp.get_pricer({
    "cusips": ["912810TZ1"],
    "timestamp": "live",
})
pr = next(iter(pricers.values())) 

print("YTM:", pr.ytm())
print("Clean price:", pr.clean_price())
print("Dirty price:", pr.dirty_price())
print("Maturity:", pr.maturity_date())
print("Meta:", pr.meta())

YTM: 4.955
Clean price: 94.63257814281297
Dirty price: 95.10495383342071
Maturity: 2044-02-15
Meta: {'record_date': datetime.date(2024, 2, 29), 'label': 'T 4 1/2 Feb 44', 'cusip': '912810TZ1', 'oi': '20-Year', 'auction_date': datetime.date(2024, 2, 21), 'issue_date': datetime.date(2024, 2, 29), 'maturity_date': datetime.date(2044, 2, 15), 'cpn': 4.5, 'rank': 8, 'timestamp': datetime.datetime(2026, 3, 24, 14, 0, 55, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>)}
